In [1]:
import numpy as np 
import pandas as pd 
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Ignorer tous les avertissements (utilisez avec précaution)
warnings.filterwarnings("ignore")

In [2]:
file_path = "DATASET_stage_total 2.0.xlsm"
sheet_name = 'TAB'
data = pd.read_excel(file_path, sheet_name=sheet_name)

In [3]:
seuil_dry = lambda x:0.8 if x<10 else (1.6 if x<20 else 5)
seuil_ver = lambda x:0.8 if x<10 else (1.8 if x<20 else 5)
seuil_slh = lambda x:0.5 if x<3 else (1.35 if x<10 else (2 if x<20 else 5))
seuil_glace = lambda x:0.6 if x<10 else (1.5 if x<20 else 5)
seuil_ncg = lambda x:0.4 if x<3 else (0.9 if x<10 else (1.5 if x<20 else 5))
seuil_ncb = lambda x:0.25 if x<3 else (0.7 if x<10 else (1.3 if x<20 else 5))
seuil_ncc = lambda x:0.4 if x<3 else (0.8 if x<10 else (1.5 if x<20 else 5))
seuil_nfb = lambda x:0.5 if x<3 else (0.8 if x<10 else (1.4 if x<20 else 5))
seuil_nfg = lambda x:0.4 if x<3 else (0.8 if x<10 else (1.4 if x<20 else 5))


In [4]:
df = data.copy()
df = df.dropna(axis = 0)
df = df[df.Contaminant != '-']
df.loc[df['Moyenne CFL C'] < 0, 'Moyenne CFL C'] = 0
df["180 - Fv"] = (180 - df['Fv (kg)']).abs()


In [5]:
def filtrer(row):
    if row.Contaminant == 'Verglas':
        if (row["Ic G%"] > seuil_ver(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 20)  | (row["180 - Fv"] > 6.1):
            return "refus"
        else:
            return "Valide"
            
    elif row.Contaminant == 'DRY':
        if (row["Ic G%"] > seuil_dry(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 15)  | (row["180 - Fv"] > 5):
            return "refus"
        else:
            return "Valide"
        
    elif row.Contaminant == 'SLUSH':
        if (row["Ic G%"] > seuil_slh(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 28.5)  | (row["180 - Fv"] > 6):
            return "refus"
        else:
            return "Valide"
            
    elif row.Contaminant == 'Glace':
        if (row["Ic G%"] > seuil_glace(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 15)  | (row["180 - Fv"] > 5):
            return "refus"
        else:
            return "Valide"
            
    elif row.Contaminant == 'N.C. (G)':
        if (row["Ic G%"] > seuil_ncg(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 18)  | (row["180 - Fv"] > 6):
            return "refus"
        else:
            return "Valide"
            
    elif row.Contaminant == 'N.C. (B)':
        if (row["Ic G%"] > seuil_ncb(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 15)  | (row["180 - Fv"] > 5):
            return "refus"
        else:
            return "Valide"
            
    elif row.Contaminant == "NC.C.":
        if (row["Ic G%"] > seuil_ncc(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 16)  | (row["180 - Fv"] > 5):
            return "refus"
        else:
            return "Valide"
            
    elif row.Contaminant == "Neige fraiche (B)":
        if (row["Ic G%"] > seuil_nfb(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 15)  | (row["180 - Fv"] > 5):
            return "refus"
        else:
            return "Valide"
            
    elif (row["Ic G%"] > seuil_nfg(row["Moyenne G% [%]"])) | (row["Ic Fv"] > 15)  | (row["180 - Fv"] > 5):
        return "refus"
    else:
        return "Valide"


In [6]:
df["Test_2"] = df.apply(filtrer, axis = 1)

In [7]:
df[df.Test_2 == "Valide"].Contaminant.value_counts()

Contaminant
NC.C.                252
Neige fraiche (B)    196
N.C. (B)             148
Neige fraiche (G)    132
Glace                131
N.C. (G)             123
SLUSH                 85
Verglas               60
DRY                   19
Name: count, dtype: int64

Apres avoir filter cette base de donnée on s'attend à avoir:
- 60 le nombre des données gardée pour le contaminant Verglas
- 19 le nombre des données gardée pour le contaminant Dry
- 85 le nombre des données gardée pour le contaminant Slush
- 131 le nombre des données gardée pour le contaminant Glace
- 123 le nombre des données gardée pour le contaminant N.C. (G)
- 148 le nombre des données gardée pour le contaminant N.C. (B)
- 252 le nombre des données gardée pour le contaminant NC.C.
- 196 le nombre des données gardée pour le contaminant Neige fraiche (B)
- 132 le nombre des données gardée pour le contaminant Neige fraiche (G)

C'est à dire à un nombre total des données validées de 1146, ce qui est conforme à ce qu'on a obtenue dans la cellule suivante.  
Valider 1146 observations, signifie un refus de 987 observations ce qui constitue un peu prés la moitié de notre base de données initiale, mais comme toujours: "La qualité de la base de données est plus importante que sa quantité."

In [20]:
pd.concat([df.Test_2.value_counts(),df.Test_2.value_counts(True)*100], axis = 1).rename(columns = {"count" : "Quantité", "proportion": "Proportions %"})

,Quantité,Proportions %
Test_2,,
Valide,1146,53.727145
refus,987,46.272855


In [ ]:
df.to_csv("Data_after_filtering-3.csv", index=False)